# Wie lange dauert es ein großes LP zu lösen?

* Wir erzeugen ein zufälliges Transportproblem wie im Skript beschrieben. Das LP ist $$\begin{align}\min\sum_{i\in\mathcal B,j\in\mathcal C}(b_i&+c_{ij}x_{ij})\\ \text{s.t. } \sum_{j\in\mathcal C}x_{ij}&\leq a_i \text{ für alle }i\in \mathcal B\\ \sum_{i\in\mathcal B}x_{ij} &= d_j\text{ für alle }j\in\mathcal C\\ x&\geq 0\end{align}$$
* Wir wählen die Größe von $\mathcal B$ und $\mathcal C$ groß, so dass wir viele Variablen und Bedingungen haben und messen die Zeit, die nötig ist, um das LP zu lösen. Um ein wenig statistische Sicherheit zu erhalten, wiederholen wir den Versuch mehrfach.

Wir starten mit ein paar Standard-imports.

In [1]:
import numpy as np # wissenschaftliches Rechnen
import mip # das MIP/LP-Paket
import time # zur Zeitmessung

# Zufallsgenerator
rng = np.random.default_rng()

Als nächstes definieren wir eine Methode, die eine Zufallsinstanz erzeugt, und eine Methode, die das LP-Modell aufsetzt.

In [2]:
def rnd_inst(B,C):
    # try at most twenty times to generate feasible instance
    # to make this more likely, scale offer 
    for _ in range(20):
        # production cost
        b=rng.random(size=B)
        # transport cost
        c=rng.random(size=(B,C))
        # offer
        a=rng.random(size=B) * C/B
        # demand
        d=rng.random(size=C)
        if sum(a)>=sum(d):
            return b,c,a,d
    raise Exception("could not generate feasible instance!")

def setup_model(b,c,a,d):
    B=len(a)
    C=len(d)
    model = mip.Model()
    # variables non-negative by default
    x=np.array([[model.add_var() for _ in range(C)] for _ in range(B)])
    
    model.objective=mip.minimize(mip.xsum(x[i,j]*(c[i,j]+b[i]) for i in range(B) for j in range(C)))
    for i in range(B):
        model += mip.xsum( x[i,j] for j in range(C) )<=a[i]
    
    for j in range(C):
        model +=mip.xsum(x[i,j] for i in range(B))==d[j]
    return model

In [3]:
repeats=3 # Anzahl Wiederholungen
B=1000 # Anzahl Fabriken
C=1000 # Anzahl Kunden
failed=0

times=[] # die Laufzeiten
obj_values=[] # wir erheben den optimalen Wert nur, um sicher zu gehen, dass tatsächlich sinnvolle Lösungen berechnet werden

total_start=time.time()
print("löse Instanz...",end="")
for i in range(repeats):
    print(f"{i+1}..",end="")
    try:
        model=setup_model(*rnd_inst(B,C))
    except:
        failed=failed+1
    start=time.time()
    model.verbose = 0 # suppress output from solver
    model.optimize()
    running_time=time.time()-start
    times.append(running_time)
    obj_values.append(model.objective_value)
print(f"\nfertig nach {round(time.time()-total_start)}s")

löse Instanz...1..2..3..
fertig nach 129s


In [4]:
times,obj_values

([37.588231563568115, 32.8481707572937, 36.62001585960388],
 [254.5814143623925, 244.0548806221428, 244.2614967936663])

In [5]:
print(f"Anzahl Variablen..........: {B*C}")
print(f"Anzahl Ungleichungen......: {B+C}")
avg_running_time=np.mean(times)
print(f"Durchschnittliche Laufzeit: {round(avg_running_time,1)}s")

Anzahl Variablen..........: 1000000
Anzahl Ungleichungen......: 2000
Durchschnittliche Laufzeit: 35.7s
